# Notebook 03 — Statcast Data Pull (Baseball Savant)

Source: `baseballsavant.mlb.com` custom leaderboard CSV export  
Rate limit: 1 s between calls, browser-like User-Agent  

This notebook pulls two Statcast leaderboards for the 2025 season:

1. Hitting — xBA, xSLG, xwOBA, xOBP, xISO, avg_swing_speed, fast_swing_rate, blasts_contact, blasts_swing, squared_up_contact, squared_up_swing, avg_swing_length, swords, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path, exit_velocity_avg  launch_angle_avg, sweet_spot_percent, barrel_batted_rate
2. Pitching — xERA, xBA, xSLG, xwOBA, xOBP, xISO, wOBA, k_percent, bb_percent, whiff_percent, chase_percent, exit_velocity_avg  sweet_spot_percent, barrel_batted_rate, hard_hit_percent, groundballs_percent  

Join key is `mlbam_id` (the `player_id` column in Savant's CSV).

In [32]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from scrapers.baseball_savant import get_statcast_hitting, get_statcast_pitching
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Statcast Hitting Leaderboard
### What each metric measures and why it's more predictive

| Statcast Metric | Group | What it is |
|---|---|---|
| **xBA** (Expected Batting Average) | Expected Outcomes | Built from exit velocity and launch angle on every batted ball. Removes BABIP noise from batting average. Year-to-year correlation is ~0.75 vs. ~0.60 for AVG. |
| **xSLG** (Expected Slugging) | Expected Outcomes | Same EV/LA model applied to extra-base power. Reflects true contact quality regardless of park effects or whether a fly ball carried. |
| **xwOBA** (Expected Weighted On-Base Average) | Expected Outcomes | Combines contact quality with walk/strikeout rates on a run-value scale. More stable than actual wOBA because defense and park are removed from batted-ball outcomes. |
| **xOBP** (Expected On-Base Percentage) | Expected Outcomes | Applies the expected model to hit outcomes while keeping real BB/HBP. Removes BABIP luck from the hits component. |
| **xISO** (Expected Isolated Power) | Expected Outcomes | Expected extra-base power per at-bat. Mirrors traditional ISO but uses the EV/LA model to separate true power from park and luck. |
| **Avg Swing Speed** | Bat Tracking | Average bat speed measured 6 inches from the barrel across competitive swings. Faster swings create more potential exit velocity. |
| **Fast Swing Rate** | Bat Tracking | Percentage of swings at or above 75 mph bat speed. Shows how often a hitter generates elite swing speed. |
| **Blasts (Contact %)** | Bat Tracking | Percentage of contact events classified as a "blast": a swing that is both fast and squared up. Strong single-metric for hard contact production. |
| **Blasts (Swing %)** | Bat Tracking | Percentage of all swings that produce blast contact. Lower than blasts_contact because it includes swings that miss entirely. |
| **Squared-Up Contact %** | Bat Tracking | Percentage of contact where the batter reached close to the maximum possible exit velocity given bat speed and pitch speed. Tracks contact precision. |
| **Squared-Up Swing %** | Bat Tracking | Percentage of all swings (including misses) that result in squared-up contact. Paired with whiff rate to gauge overall swing efficiency. |
| **Avg Swing Length** | Bat Tracking | Average distance the barrel travels from start to contact in XYZ space (feet). Shorter paths mean quicker swings and less vulnerability to inside pitches. |
| **Swords** | Bat Tracking | Count of swings resulting in an extreme miss. The batter is beaten badly enough that the swing looks awkward. High sword counts signal vulnerability to quality stuff. |
| **Attack Angle** | Bat Tracking | Vertical angle of the bat at contact (degrees). Positive = upward swing plane. Optimal range (~5–15°) aligns with average pitch descent to maximize contact area. |
| **Attack Direction** | Bat Tracking | Horizontal swing direction at contact relative to home plate. Reflects pull tendency vs. opposite-field approach in the swing mechanics, independent of batted-ball outcomes. |
| **Ideal Angle Rate** | Bat Tracking | Percentage of swings where attack angle falls in the optimal launch window. Higher rates mean the swing plane consistently matches pitch trajectory. |
| **Vertical Swing Path** | Bat Tracking | Steepness of the swing plane from start to contact. Works alongside attack angle to describe the full 3D geometry of the swing. |
| **Exit Velocity (Avg)** | Batted Ball Outcomes | Average speed off the bat across all batted balls. Direct product of swing speed and contact quality. Strong predictor of wOBA and HR rate. |
| **Launch Angle (Avg)** | Batted Ball Outcomes | Average vertical angle of the batted ball. The 10–30° range produces the most run value. Explains why two hitters with the same EV can have very different results. |
| **Sweet Spot %** | Batted Ball Outcomes | Percentage of batted balls hit between 8° and 32° launch angle. Balls in this window have the highest hit probability and slugging potential. |
| **Barrel %** | Batted Ball Outcomes | Percentage of batted balls meeting the optimal EV/LA threshold (≥98 mph, 26–30°). Barrels carry a ~1.000 xSLG and are the most consistent power indicator year over year.|

In [34]:
hitting = get_statcast_hitting(2025)
print(f"Shape: {hitting.shape}")
print(f"\nDtypes:\n{hitting.dtypes}")
print(f"\nNull rates:\n{hitting.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
hitting.head(10)

Shape: (537, 23)

Dtypes:
mlbam_id                 int64
name                    object
xBA                    float64
xSLG                   float64
xwOBA                  float64
xOBP                   float64
xISO                   float64
avg_swing_speed        float64
fast_swing_rate        float64
blasts_contact         float64
blasts_swing           float64
squared_up_contact     float64
squared_up_swing       float64
avg_swing_length       float64
swords                   int64
attack_angle           float64
attack_direction       float64
ideal_angle_rate       float64
vertical_swing_path    float64
exit_velocity_avg      float64
launch_angle_avg       float64
sweet_spot_percent     float64
barrel_batted_rate     float64
dtype: object

Null rates:
mlbam_id               0.0
name                   0.0
xBA                    0.0
xSLG                   0.0
xwOBA                  0.0
xOBP                   0.0
xISO                   0.0
avg_swing_speed        0.0
fast_swing_rate   

,mlbam_id,name,xBA,xSLG,xwOBA,xOBP,xISO,avg_swing_speed,fast_swing_rate,blasts_contact,blasts_swing,squared_up_contact,squared_up_swing,avg_swing_length,swords,attack_angle,attack_direction,ideal_angle_rate,vertical_swing_path,exit_velocity_avg,launch_angle_avg,sweet_spot_percent,barrel_batted_rate
0,666163,"Rortvedt, Ben",0.185,0.246,0.240,0.270,0.062,68.8,4.9,8.1,5.9,22.3,16.1,7.0,2,10.8,-0.9,59.7,28.9,81.2,13.6,23.9,3.3
1,690022,"Ritter, Ryan",0.211,0.276,0.248,0.270,0.066,69.1,6.7,7.1,4.7,26.1,17.2,6.7,12,10.4,1.1,41.0,35.2,85.2,7.4,30.5,1.5
2,602104,"Urías, Ramón",0.227,0.375,0.289,0.283,0.148,70.8,12.5,14.2,10.8,34.2,25.9,7.7,15,13.7,0.3,70.6,27.1,89.2,13.2,33.6,8.0
3,680779,"Davis, Henry",0.213,0.389,0.293,0.279,0.176,73.9,39.3,14.2,10.3,26.5,19.3,7.3,8,8.6,-7.5,55.1,26.3,88.6,23.3,33.0,11.5
4,700246,"Williams, Carson",0.169,0.313,0.233,0.217,0.144,73.6,33.1,9.2,5.6,21.1,12.9,7.6,3,15.3,-2.7,50.0,34.1,85.9,8.0,30.4,8.9
5,682622,"Marte, Noelvi",0.245,0.418,0.304,0.285,0.173,73.3,33.2,15.7,11.7,32.5,24.4,7.4,15,9.7,-4.3,62.3,31.0,88.5,12.7,32.0,9.0
6,683090,"Lugo, Matthew",0.217,0.379,0.260,0.228,0.162,73.4,35.2,18.3,12.0,38.0,25.0,7.4,1,9.3,-1.6,49.1,35.5,88.2,12.9,33.3,13.3
7,694671,"Langford, Wyatt",0.239,0.448,0.346,0.344,0.210,73.1,29.0,17.9,13.4,34.4,25.8,7.0,10,16.8,-3.1,48.6,33.7,91.4,17.5,35.9,14.0
8,662139,"Varsho, Daulton",0.226,0.494,0.327,0.279,0.267,75.6,56.2,15.0,10.6,28.8,20.4,7.9,5,12.9,-3.8,62.4,29.8,89.9,23.5,36.9,15.9
9,656577,"Jackson, Alex",0.178,0.362,0.270,0.252,0.185,76.1,62.8,17.9,11.6,24.1,15.7,7.8,2,11.7,-8.7,51.2,30.8,88.3,14.3,40.7,14.8


## 2. Statcast Pitching Leaderboard
### What each metric measures and why it's more predictive

| Statcast Metric | Group | What it is |
|---|---|---|
| **xERA** (Expected ERA) | Expected Outcomes | ERA built from contact quality allowed (EV + LA model) rather than actual runs. Removes defense, sequencing, and strand-rate luck, making it more park- and defense-neutral than standard ERA estimators. |
| **xBA** (Expected Batting Average Allowed) | Expected Outcomes | Expected hit rate against, based on the EV/LA of every batted ball the pitcher allowed. Removes BABIP noise. A pitcher with low xBA but high BA is likely benefiting from good defense or sequencing. |
| **xSLG** (Expected Slugging Allowed) | Expected Outcomes | EV/LA model applied to extra-base contact allowed. Reflects true power damage regardless of park or whether fly balls happened to carry. |
| **xwOBA** (Expected wOBA Allowed) | Expected Outcomes | Combines contact quality with real strikeouts and walks on a run-value scale. More stable than actual wOBA allowed because defense is removed from batted-ball outcomes. Widely considered the strongest single contact-quality read on a pitcher. |
| **xOBP** (Expected OBP Allowed) | Expected Outcomes | Applies the expected model to hit outcomes while keeping real BB/HBP allowed. Distinguishes between high OBP allowed driven by walks versus hit luck. |
| **xISO** (Expected ISO Allowed) | Expected Outcomes | Expected extra-base power allowed per at-bat. Separates true power suppression from park effects and HR/FB luck. |
| **wOBA** (Weighted On-Base Average Allowed) | Expected Outcomes | Actual run-value-weighted on-base average allowed. Included alongside xwOBA so the gap between the two can be tracked as a luck indicator. |
| **K%** (Strikeout Rate) | Plate Discipline | Strikeouts per plate appearance. Defense plays no role in strikeouts, which is why K% holds up better than most pitching stats year over year and is a strong predictor of ERA estimators. |
| **BB%** (Walk Rate) | Plate Discipline | Walks per plate appearance. Free baserunners with no defense involved. Combined with K% into K-BB%, it gives the clearest read on a pitcher's command. |
| **Whiff %** | Plate Discipline | Swings and misses divided by total swings. Tracks swing-and-miss ability on contact attempts specifically, making it more granular than SwStr%, which includes pitches batters never offered at. |
| **Chase %** (OZ Swing %) | Plate Discipline | Percentage of pitches outside the zone that batters swing at. Higher rates mean hitters are making poor decisions, usually a function of pitch quality and location at the edges. |
| **Exit Velocity (Avg Allowed)** | Batted Ball Outcomes | Average EV on all batted balls against. Shows how hard the pitcher is being hit. Less park-dependent than ERA or HR rate. |
| **Sweet Spot % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against landing in the 8–32° launch angle window. Balls in this range have the highest hit probability and slugging potential. Keeping this rate low means the pitcher is generating weak or mis-hit contact. |
| **Barrel % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against meeting the barrel threshold (≥98 mph, 26–30° LA). Barrels carry ~1.000 xSLG against and this rate is one of the more consistent damage-allowed metrics year over year. |
| **Hard-Hit % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against at ≥95 mph exit velocity. Broader than barrel rate, it picks up overall contact damage including line drives that don't meet the barrel angle threshold. |
| **GB% (Allowed)** | Batted Ball Outcomes | Ground ball rate on batted balls against. Ground balls have the lowest xSLG of any batted ball type and can't leave the park. High GB% pitchers are more defense-dependent but limit big innings. |

In [36]:
pitching = get_statcast_pitching(2025)
print(f"Shape: {pitching.shape}")
print(f"\nDtypes:\n{pitching.dtypes}")
print(f"\nNull rates:\n{pitching.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
pitching.head(10)

Shape: (657, 18)

Dtypes:
mlbam_id                 int64
name                    object
xERA                   float64
xBA                    float64
xSLG                   float64
xwOBA                  float64
xOBP                   float64
xISO                   float64
wOBA                   float64
k_percent              float64
bb_percent             float64
whiff_percent          float64
chase_percent          float64
exit_velocity_avg      float64
sweet_spot_percent     float64
barrel_batted_rate     float64
hard_hit_percent       float64
groundballs_percent    float64
dtype: object

Null rates:
mlbam_id               0.0
name                   0.0
xERA                   0.0
xBA                    0.0
xSLG                   0.0
xwOBA                  0.0
xOBP                   0.0
xISO                   0.0
wOBA                   0.0
k_percent              0.0
bb_percent             0.0
whiff_percent          0.0
chase_percent          0.0
exit_velocity_avg      0.0
sweet_spot_

,mlbam_id,name,xERA,xBA,xSLG,xwOBA,xOBP,xISO,wOBA,k_percent,bb_percent,whiff_percent,chase_percent,exit_velocity_avg,sweet_spot_percent,barrel_batted_rate,hard_hit_percent,groundballs_percent
0,607200,"Fedde, Erick",5.41,0.275,0.461,0.355,0.357,0.186,0.344,13.3,10.8,18.0,24.8,90.2,35.3,7.9,42.6,42.1
1,669358,"Baz, Shane",3.88,0.230,0.386,0.306,0.308,0.157,0.330,24.8,9.0,25.8,29.3,89.0,32.3,9.5,39.4,47.0
2,571760,"Heaney, Andrew",5.66,0.284,0.492,0.362,0.349,0.208,0.357,16.0,7.5,19.9,29.0,90.2,37.7,10.2,43.6,37.9
3,641816,"Mahle, Tyler",4.24,0.253,0.413,0.319,0.316,0.160,0.265,19.1,8.4,23.2,27.9,88.5,37.8,8.0,37.1,38.6
4,690916,"Fitts, Richard",5.62,0.266,0.488,0.361,0.342,0.221,0.337,20.5,8.2,25.3,32.8,90.7,39.6,12.7,48.5,43.3
5,554340,"García, Yimi",4.77,0.242,0.417,0.336,0.353,0.175,0.255,27.8,13.3,30.7,23.5,91.1,38.5,9.6,40.4,44.2
6,681892,"Funderburk, Kody",3.66,0.244,0.340,0.298,0.328,0.097,0.321,21.9,9.8,21.4,24.0,90.4,29.5,4.9,42.6,54.1
7,687849,"Kent, Zak",3.41,0.234,0.302,0.288,0.325,0.068,0.305,21.1,10.5,22.9,21.2,87.2,39.2,0.0,29.4,37.3
8,623167,"Flexen, Chris",5.09,0.275,0.479,0.346,0.323,0.203,0.285,12.4,6.7,16.7,26.0,90.9,34.7,9.0,41.7,36.8
9,595345,"Okert, Steven",2.61,0.185,0.326,0.253,0.249,0.141,0.233,30.4,6.9,32.8,27.3,88.4,29.2,7.0,32.7,25.7


## 3. Null Rate Summary (both DataFrames)

In [38]:
null_summary = pd.DataFrame({
    "hitting": hitting.isnull().mean().round(4),
    "pitching": pitching.isnull().mean().round(4),
}).fillna("—")
print("=== Null Rate Summary ===")
null_summary

=== Null Rate Summary ===


,hitting,pitching
attack_angle,0.0,—
attack_direction,0.0,—
avg_swing_length,0.0,—
avg_swing_speed,0.0,—
barrel_batted_rate,0.0,0.0
bb_percent,—,0.0
blasts_contact,0.0,—
blasts_swing,0.0,—
chase_percent,—,0.0
exit_velocity_avg,0.0,0.0


## 4. Save to Parquet


In [40]:
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

hitting.to_parquet(os.path.join(data_dir, "raw_statcast_hitting.parquet"), index=False)
pitching.to_parquet(os.path.join(data_dir, "raw_statcast_pitching.parquet"), index=False)

print("Saved:")
for f in ["raw_statcast_hitting.parquet", "raw_statcast_pitching.parquet"]:
    path = os.path.join(data_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f} — {size_kb:.1f} KB")

Saved:
  raw_statcast_hitting.parquet — 53.5 KB
  raw_statcast_pitching.parquet — 49.8 KB
